# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

print(f"Dataset name: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in the dataset are always referenced by their `@id` field. Below we print the available Record Sets with their `@id` values and a summary of fields for each.

In [ ]:
# List available record sets and fields
print("Available record sets in the dataset:")
for rs in dataset.metadata.record_sets:
    print(f"Record Set @id: {rs['@id']}, Name: {rs.get('name', '')}")
    print("  Fields:")
    for field in rs['fields']:
        print(f"    Field @id: {field['@id']:35s} | Name: {field.get('name','')} | Data type: {field.get('dataType','')}")
    print("")

You can also print a few records from a record set for preview. 

Below is an example for the first available record set (replace with any `@id` as needed):

In [ ]:
# Preview a few records for the first record set
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
if record_sets_ids:
    sample_rs = record_sets_ids[0]
    print(f"Preview of first 2 records in RecordSet {sample_rs}:")
    for idx, rec in enumerate(dataset.records(record_set=sample_rs)):
        print(rec)
        if idx >= 1:
            break
else:
    print("No record sets defined in the dataset.")

## 3. Data Extraction
Load data from all specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview. Entities are referenced by their `@id` field.

In [ ]:
# Extract data from each record set
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set in record_sets:
    print(f"Loading records for RecordSet @id: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if records:
        dataframes[record_set] = pd.DataFrame(records)

# Example: show the columns for the first record set
if record_sets:
    print(f"\nColumns in RecordSet {record_sets[0]}:")
    print(dataframes[record_sets[0]].columns.tolist())
    display(dataframes[record_sets[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, or grouping. All field and record set references use their `@id`.

> **Note:** Replace `<numeric_field_id>` and `<group_field_id>` with an actual `@id` from the previously enumerated fields in your chosen record set.

In [ ]:
# Example for EDA on the first record set using a numeric field

# Assign your relevant record set and field IDs here
record_set_id = record_sets[0] if record_sets else None
numeric_field_id = None
group_field_id = None

# Try to auto-select a numeric field
if record_set_id:
    df = dataframes[record_set_id]
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Optionally a group field
    for col in df.columns:
        if col != numeric_field_id and pd.api.types.is_object_dtype(df[col]):
            group_field_id = col
            break
    print(f"Selected numeric field for EDA: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}")
else:
    print("No record sets/data found.")

if record_set_id and numeric_field_id:
    threshold = df[numeric_field_id].mean()  # Use a data-dependent threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouped mean
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All references use valid field and record set `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id} in RecordSet {record_set_id}')
    plt.show()
    
    # Boxplot by group if possible
    if group_field_id:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from this dataset exploration.

- Loaded metadata and records from the Croissant dataset with entity references by `@id`.
- Previewed available record sets and fields, and dynamically loaded data for analysis.
- Performed basic EDA including filtering, normalization, and grouping by selected fields.
- Visualized distributions highlighting the characteristics of numeric features across groups.

This notebook can be extended to support more detailed analysis and to address specific research questions relevant to protein abundance and extracellular vesicles.